# Structured Streaming - Resumo para Certificação DEA

## 📊 Conceito Fundamental

**Data Stream** = qualquer fonte de dados que cresce ao longo do tempo:
- Novos arquivos chegando no cloud storage
- Atualizações capturadas via CDC (Change Data Capture)
- Eventos em filas de mensagens (Kafka, Event Hubs)

**Abordagens de processamento:**
| Abordagem | Descrição |
|-----------|-----------|
| Reprocessamento total | Processa toda a fonte a cada execução |
| **Structured Streaming** | Processa apenas dados novos desde a última atualização |

> 💡 **Conceito-chave para prova:** O Structured Streaming trata dados infinitos como uma **tabela ilimitada (unbounded table)**, permitindo usar APIs familiares do Spark para dados em streaming.

---

## 🔄 Arquitetura Básica

![Spark Streaming Workflow](https://www.databricks.com/sites/default/files/2020/04/gsasg-spark-streaming-workflow.png)

**Fontes comuns no Azure Databricks:**
- Arquivos em cloud storage (via **Auto Loader**)
- Message buses (Kafka, Event Hubs)
- **Delta Lake** (streaming reads/writes)

---

## 📖 DataStreamReader

```python
streamDF = spark.readStream \
    .table("Input_Table")

# Ou com Auto Loader (cloudFiles)
streamDF = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .load("/path/to/files")
```

---

## ✍️ DataStreamWriter

```python
streamDF.writeStream \
    .trigger(processingTime="2 minutes") \
    .outputMode("append") \
    .option("checkpointLocation", "/path/checkpoint") \
    .table("Output_Table")
```

---

## ⏱️ Trigger Intervals

| Trigger Type | Sintaxe | Comportamento |
|--------------|---------|---------------|
| **Default (No Trigger)** | Não especificado | Intervalo de 500 microsegundos |
| **Fixed Interval** | `.trigger(processingTime="2 minutes")` | Micro-batch inicia no intervalo especificado |
| **Triggered Once** | `.trigger(once=True)` | Processa tudo em **um único batch** e para **[Deprecated]** |
| **Available Now** | `.trigger(availableNow=True)` | Processa tudo em **múltiplos micro-batches** e para |
| **Continuous** | `.trigger(continuous="2 seconds")` | Processa dados continuamente, checkpoint no intervalo **[Experimental]** |

> ⚠️ **Pegadinhas de prova:** 
> - `once=True` está **DEPRECATED** — use `availableNow=True` em seu lugar!
> - `once=True` = 1 micro-batch único | `availableNow=True` = múltiplos micro-batches
> - `continuous` é **EXPERIMENTAL** e oferece latência ultra-baixa

---

## 📤 Output Modes

| Mode | Sintaxe | Comportamento |
|------|---------|---------------|
| **Append** (default) | `.outputMode("append")` | Apenas novas linhas são adicionadas |
| **Complete** | `.outputMode("complete")` | Tabela inteira é sobrescrita a cada batch |
| **Update** | `.outputMode("update")` | Grava apenas linhas que sofreram modificações |

---

## 💾 Checkpointing

```python
.option("checkpointLocation", "/path/checkpoint")
```

**Funções do checkpoint:**
- Armazena o **estado do stream**
- Rastreia o **progresso** do processamento
- Registra **offset range** de dados processados

> 🚫 **Regra crítica:** Checkpoints **NÃO podem ser compartilhados** entre streams diferentes. Cada writer precisa de um checkpoint único!

---

## 🛡️ Garantias

| Garantia | Mecanismo |
|----------|-----------|
| **Fault Tolerance** | Checkpointing + Write-ahead logs |
| **Exactly-once** | Idempotent sinks (ex: Delta Lake) |

---

## ⛔ Operações NÃO Suportadas

Algumas operações não funcionam diretamente em streaming DataFrames:

| Operação | Motivo |
|----------|--------|
| **Sorting** | Impossível ordenar dados infinitos |
| **Deduplication** | Requer estado ilimitado |
| **Aggregations** | Requerem **windowing** e **watermarking** |

> 📚 Para agregações em streaming, use técnicas avançadas: **Windowing** e **Watermarking**

---

## 🚀 Auto Loader (cloudFiles)

**O que é:** Fonte de Structured Streaming otimizada para ingestão incremental de arquivos.

```python
# Sintaxe básica
df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", "/path/schema") \
    .load("/path/to/source")
```

**Formatos suportados:** JSON, CSV, Parquet, Avro, Text, Binary, ORC

**Modos de descoberta de arquivos:**

| Modo | Descrição | Custo |
|------|-----------|-------|
| **Directory Listing** | Lista diretórios periodicamente | Escala com nº de arquivos |
| **File Notification** | Usa eventos da cloud (Event Grid) | Menor custo para grandes volumes |

**Vantagens-chave:**
- **Schema inference** automático
- **Schema evolution** com suporte a drift
- Processa **bilhões de arquivos** eficientemente
- **Rescue column** para dados não parseados

> 🎯 **Para prova:** Auto Loader é a **recomendação oficial** para ingestão de arquivos no Databricks.

---

## 📋 Opções Importantes do Auto Loader

| Opção | Descrição |
|-------|-----------|
| `cloudFiles.format` | Formato dos arquivos (json, csv, parquet...) |
| `cloudFiles.schemaLocation` | Local para armazenar schema inferido |
| `cloudFiles.maxFilesPerTrigger` | Máximo de arquivos por micro-batch (default: 1000) |
| `cloudFiles.maxBytesPerTrigger` | Máximo de bytes por micro-batch |
| `cloudFiles.useNotifications` | Habilita modo de notificação |
| `pathGlobFilter` | Filtro de arquivos por padrão (ex: `*.png`) |
| `cloudFiles.schemaEvolutionMode` | Modo de evolução de schema |

---

## 🔍 Filtros de Arquivos (File Filters)

Para filtrar arquivos de entrada com base em um padrão específico, use a opção `pathGlobFilter`:

```python
spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "binaryFile") \
    .option("pathGlobFilter", "*.png") \
    .load("/path/to/files")
```

| Padrão | Descrição |
|--------|-----------|
| `*.png` | Apenas arquivos PNG |
| `*.json` | Apenas arquivos JSON |
| `data_*.csv` | Arquivos CSV que começam com "data_" |

> 💡 **Para prova:** `pathGlobFilter` é útil quando o diretório contém múltiplos tipos de arquivos e você quer processar apenas um tipo específico.

---

## 🔄 Schema Evolution no Auto Loader

O Auto Loader detecta automaticamente a **adição de novas colunas** nos arquivos de entrada durante o processamento.

### Configuração

```python
spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", <source_format>) \
    .option("cloudFiles.schemaEvolutionMode", <mode>) \
    .load("/path/to/files")
```

### Modos de Schema Evolution

| Modo | Comportamento |
|------|---------------|
| **`addNewColumns`** (default) | Stream **para** com `UnknownFieldException`, atualiza schema, próxima execução usa schema atualizado |
| `rescue` | Novas colunas vão para `_rescued_data` column |
| `failOnNewColumns` | Stream falha imediatamente |
| `none` | Ignora novas colunas silenciosamente |

### Fluxo do Modo Default (`addNewColumns`)

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│ Nova coluna     │ ──► │ Stream para com │ ──► │ Schema location │
│ detectada       │     │ UnknownField    │     │ é atualizado    │
└─────────────────┘     │ Exception       │     └────────┬────────┘
                        └─────────────────┘              │
                                                         ▼
                        ┌─────────────────┐     ┌─────────────────┐
                        │ Próxima execução│ ◄── │ Schema inclui   │
                        │ usa novo schema │     │ nova coluna     │
                        └─────────────────┘     └─────────────────┘
```

> ⚠️ **Importante para prova:** No modo `addNewColumns`, o Auto Loader **atualiza o schema location** automaticamente antes de lançar o erro. A próxima execução funciona com o schema atualizado!

---

## ✅ Checklist para Prova

- [ ] Structured Streaming trata dados como **tabela ilimitada**
- [ ] **Checkpoint** é obrigatório e único por stream
- [ ] `once=True` está **DEPRECATED** → use `availableNow=True`
- [ ] `once=True` → 1 micro-batch | `availableNow=True` → múltiplos micro-batches
- [ ] `continuous` trigger é **EXPERIMENTAL**
- [ ] **Append** é o output mode padrão
- [ ] Auto Loader usa formato **cloudFiles**
- [ ] Sorting e deduplication **não são suportados** diretamente
- [ ] **Exactly-once** requer sinks idempotentes
- [ ] Schema evolution é suportado pelo Auto Loader
- [ ] `pathGlobFilter` filtra arquivos por padrão (ex: `*.png`)
- [ ] Modo default de schema evolution é **`addNewColumns`**
- [ ] No `addNewColumns`, stream **para** mas **atualiza schema** automaticamente

---

## 🔗 Referências

- [Structured Streaming Concepts - Azure Databricks](https://learn.microsoft.com/en-us/azure/databricks/structured-streaming/concepts)
- [Auto Loader Documentation](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/)
- [Auto Loader Options Reference](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/options)
- [Production Considerations](https://learn.microsoft.com/en-us/azure/databricks/structured-streaming/production)